# New Keynesian Model — Python (dynare_py)

3-equation New Keynesian model:

**IS curve:**  $x_t = E_t[x_{t+1}] - \frac{1}{\sigma}(i_t - E_t[\pi_{t+1}] - r^n_t)$

**NKPC:**  $\pi_t = \beta E_t[\pi_{t+1}] + \kappa x_t$

**Taylor rule:**  $i_t = \phi_\pi \pi_t + \phi_x x_t$

**Natural rate:**  $r^n_t = \rho r^n_{t-1} + \varepsilon_t$

Variables: $y_t = [x_t, \pi_t, i_t, r^n_t]$. Written in ABC form:
$$A E_t[y_{t+1}] + B y_t + C y_{t-1} + D\varepsilon_t = 0$$

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'python'))

import numpy as np
import matplotlib.pyplot as plt
from dynare_py import DSGEModel, compute_irf, simulate, fevd

## 1. Parameters

In [ ]:
sigma = 1.0    # inverse EIS
beta  = 0.99   # discount factor
kappa = 0.1    # slope of NKPC
phi_pi = 1.5   # Taylor rule: inflation response
phi_x  = 0.5   # Taylor rule: output gap response
rho    = 0.8   # AR(1) natural rate persistence

# Variable and shock names
variables = ['x', 'pi', 'i', 'rn']
shocks    = ['eps_rn']

## 2. Build ABC matrices

Rewrite each equation as $A E_t[y_{t+1}] + B y_t + C y_{t-1} + D \varepsilon_t = 0$:

| eq | $E_t[x_{t+1}]$ | $E_t[\pi_{t+1}]$ | $E_t[i_{t+1}]$ | $E_t[r^n_{t+1}]$ | $x_t$ | $\pi_t$ | $i_t$ | $r^n_t$ | $x_{t-1}$ | $\pi_{t-1}$ | $i_{t-1}$ | $r^n_{t-1}$ | $\varepsilon_t$ |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| IS  | -1 | $1/\sigma$ | 0 | 0 | 1 | 0 | $1/\sigma$ | $-1/\sigma$ | 0 | 0 | 0 | 0 | 0 |
| NKPC | 0 | $-\beta$ | 0 | 0 | $-\kappa$ | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| TR  | 0 | 0 | 0 | 0 | $-\phi_x$ | $-\phi_\pi$ | 1 | 0 | 0 | 0 | 0 | 0 | 0 |
| AR  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | $-\rho$ | $-1$ |

In [ ]:
n, ne = 4, 1

# A: coefficients on E_t[y_{t+1}]
A = np.array([
    [-1,   1/sigma,  0,  0],   # IS
    [ 0,  -beta,     0,  0],   # NKPC
    [ 0,   0,        0,  0],   # Taylor rule (static)
    [ 0,   0,        0,  0],   # AR(1) for r^n (no forward)
])

# B: coefficients on y_t
B = np.array([
    [ 1,    0,      1/sigma,  -1/sigma],  # IS
    [-kappa, 1,     0,         0      ],  # NKPC
    [-phi_x, -phi_pi, 1,       0      ],  # Taylor rule
    [ 0,    0,      0,         1      ],  # AR(1)
])

# C: coefficients on y_{t-1}
C = np.array([
    [0, 0, 0,    0   ],
    [0, 0, 0,    0   ],
    [0, 0, 0,    0   ],
    [0, 0, 0,   -rho ],
])

# D: coefficients on eps_t
D = np.array([
    [ 0],
    [ 0],
    [ 0],
    [-1],
])

print("A (forward-looking):")
print(A)
print("\nForward-looking variables (non-zero columns of A):",
      [variables[j] for j in range(n) if np.any(A[:, j] != 0)])

## 3. Build model and solve

In [ ]:
model = DSGEModel.from_abc(A, B, C, D, variables=variables, shocks=shocks)
print(model.summary())
print("\nAugmented variables:", model.aug_variables)

In [ ]:
result = model.solve()

print("=== Blanchard-Kahn Conditions ===")
print(f"  Forward-looking equations (cols of Pi) : {result.bk.n_forward}")
print(f"  Unstable eigenvalues                   : {result.bk.n_unstable}")
print(f"  Existence  : {result.bk.existence}")
print(f"  Uniqueness : {result.bk.uniqueness}")
print(f"  BK matrix rank : {result.bk.Q2_Pi_rank} (needs {result.bk.n_unstable})")

## 4. Inspect QZ decomposition

Key transparency feature: examine the eigenvalues, QZ matrices, and the BK matrix.

In [ ]:
qz = result.qz
print("Generalised eigenvalues of G0^{-1} G1:")
for i, ev in enumerate(qz.eigenvalues):
    tag = 'UNSTABLE' if abs(ev) > 1.01 else 'stable  '
    print(f"  λ_{i} = {abs(ev):.4f}  [{tag}]")

print(f"\nStable: {qz.n_stable},  Unstable: {qz.n_unstable}")

In [ ]:
print("BK matrix Q2 @ Pi (should be invertible):")
print(np.round(result.bk.Q2_Pi, 4))
print(f"Singular values: {np.linalg.svd(result.bk.Q2_Pi, compute_uv=False).round(4)}")

In [ ]:
print("Stable transition T_hat = BB11^{-1} AA11 (in transformed Schur space):")
print(np.round(result.T_hat, 4))
print(f"Eigenvalues of T_hat: {np.sort(np.abs(np.linalg.eigvals(result.T_hat)))[::-1].round(4)}")

In [ ]:
print("Policy matrix T (in original space):")
aug_vars = model.aug_variables
import pandas as pd
print(pd.DataFrame(result.T.round(4), index=aug_vars, columns=aug_vars))

## 5. Impulse Response Functions

In [ ]:
periods = 40
irf = compute_irf(
    result, shock_index=0, shock_size=1.0,
    periods=periods, var_names=aug_vars
)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()

# Show original variables only (not _fwd counterparts)
orig_vars = model.variables
for ax, var in zip(axes, orig_vars):
    ax.plot(irf[var].values, 'b-o', markersize=3)
    ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
    ax.set_title(var, fontsize=12)
    ax.set_xlabel('Periods')

for ax in axes[len(orig_vars):]:
    ax.set_visible(False)

fig.suptitle('IRF to 1 s.d. natural rate shock ($r^n$)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Stochastic simulation

In [ ]:
sim = simulate(result, periods=500, burn=100, seed=42, var_names=aug_vars)

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
axes = axes.flatten()
for ax, var in zip(axes, orig_vars):
    ax.plot(sim[var].values, 'b', linewidth=0.8)
    ax.set_title(var)
    ax.set_xlabel('Periods')
plt.suptitle('Stochastic simulation', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Forecast Error Variance Decomposition

In [ ]:
decomp = fevd(result, horizon=20, shock_names=shocks, var_names=aug_vars)

print("Variance (output gap x) by horizon:")
print(decomp['variance'][['x']].head(10).round(6))

## 8. Deep decomposition: how Z1 and T_hat drive the dynamics

The policy matrix $T = Z_1 \hat{T} Z_1^\top$ where $\hat{T}=B_{11}^{-1}A_{11}$ is the stable block in Schur space.  
The columns of $Z_1$ show which directions in variable space correspond to stable modes.

In [ ]:
Z1 = result.Z1
print("Z1 (n × n_stable) — stable mode directions in original variable space:")
print(pd.DataFrame(Z1.round(4), index=aug_vars, columns=[f'mode_{i}' for i in range(Z1.shape[1])]))

In [ ]:
print("Eta matrix (expectational errors from shocks):")
print("eta_t = eta_matrix @ eps_t")
print(pd.DataFrame(result.eta_matrix.round(6),
      index=[f'eta_{v}' for v in [variables[j] for j in range(n) if np.any(A[:, j] != 0)]],
      columns=shocks))